# End-to-end orchestration

Run the **whole** disruption pipeline and inspect the shared `GraphState` **after each hop**, so you can see how every agent hands work to the next.

Start here to understand the system; then open a per-agent notebook (`10_classify`, `15_news`, `20_impact`, `30_forecast`, `40_simulate`, `50_recommend`) to work on one node.

## Overall architecture

```mermaid
flowchart LR
    subgraph ingest["Ingestion service (out of band)"]
        ISVC["connectors → normalize ·<br/>gate · dedupe → Postgres signals"]
    end

    subgraph pipeline["LangGraph pipeline"]
        ENTRY["ingestion_node →<br/>guardrail → seed"] --> NEWS["news"]
        NEWS --> WX["weather"] --> CLS["classify"]
        CLS --> IMP["impact"] --> FC["forecast"]
        FC --> SIM["simulate"] --> REC["recommend"]
    end

    ISVC -->|"status = 'new' rows"| ENTRY
    REC --> OUT["GraphState<br/>render: CLI / Gradio"]
```

The ingestion service writes signals to Postgres out of band; the graph reads the new rows (`ingestion_node`), guards them, tops up with a synthetic signal when there are none (`seed_node`), enriches each signal through `news` and `weather`, then runs classification, impact mapping, forecasting, simulation, and mitigation recommendation. Every node reads and writes the typed `GraphState` channels.

## Setup — Postgres + ingest signals

Start Postgres from the **repo root** (`docker compose up -d postgres`), then these cells
wait for it and load real signals so the rest of the notebook runs against live data.
**Prerequisite:** Docker Desktop running.

> Prefer no database? Skip this section — the agent notebooks run fully offline on
> synthetic sample state.

In [ ]:
# Postgres is managed by `docker compose` from the REPO ROOT (where `.env` lives).
# Start it there first — in a terminal at the repo root:
#     docker compose up -d postgres
# (Already ran `docker compose up`? It's running — just continue.)
# The next cell waits until the database is reachable.
print("Ensure Postgres is up: run `docker compose up -d postgres` from the repo root.")

In [ ]:
import time

from agentic_scd.config import get_settings
from agentic_scd.db import ping

# get_settings() is cached; clear it each poll so a freshly-started DB is picked up.
deadline = time.time() + 60
get_settings.cache_clear()
status = ping()
while not status and time.time() < deadline:
    time.sleep(2)
    get_settings.cache_clear()
    status = ping()

print(status.detail)
if not status:
    print(
        "\n[!] Postgres isn't reachable. Is Docker Desktop running, and did "
        "`docker compose up -d postgres` succeed? You can still run the agent "
        "notebooks offline on synthetic sample state."
    )

In [ ]:
# Seed historical baselines (Freightos snapshot + Kaggle SupplyChainNet) — one-shot.
!uv run agentic-scd-batch

In [ ]:
# Run every enabled connector once (RSS + Open-Meteo + synthetic) through the pipeline.
!uv run agentic-scd-collect

In [ ]:
import psycopg

from agentic_scd.db import DatabaseNotConfiguredError, connect

try:
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            "SELECT status, count(*) FROM signals GROUP BY status ORDER BY status"
        )
        rows = cur.fetchall()
    if rows:
        print("signals by status:")
        for status_value, n in rows:
            print(f"  {status_value:>12}: {n}")
    else:
        print("signals table is empty — re-run the collect/batch cells above.")
except (DatabaseNotConfiguredError, psycopg.OperationalError) as exc:
    print(f"No DB available ({exc}); skipping the row-count check.")

## Step through the graph

`build_graph()` compiles the LangGraph pipeline. Streaming in `"updates"` mode yields each node's state delta as it runs, so we can watch the handoff. With no DB (or no new rows), `seed_node` injects a synthetic signal so the chain always produces a full result.

In [ ]:
from agentic_scd.graph import build_graph

graph = build_graph()
state: dict = {}
for update in graph.stream({}, stream_mode="updates"):
    for node, delta in update.items():
        state.update(delta)
        changed = ", ".join(delta) if delta else "(no change)"
        print(f"> {node:<18} updated: {changed}")

### Signals that entered the chain

In [ ]:
for s in state["new_signals"]:
    print(f"[{s.source_type}] {s.title}")

### After `news` — extracted event type, region, and entities

In [ ]:
for item in state["event_analyses"]:
    entities = ", ".join(item.entities) or "-"
    print(f"{item.event_type:>24}  region={item.extracted_region or '-':<16} entities={entities}")

### After `weather` — weather-linked disruption monitoring

In [ ]:
if state.get("weather_risks"):
    for item in state["weather_risks"]:
        print(f"{item.region or '-':<16} alert={item.alert_level:<8} severity={item.severity_score:.1f} summary={item.summary}")
else:
    print("No weather-linked disruption escalated from this batch.")

### After `classify` — category + risk score per signal

In [ ]:
for c in state["classifications"]:
    print(f"{c.category:>16}  risk={c.risk_score:.2f}  ({c.rationale})")

### After `impact` — affected parts of our network

In [ ]:
for im in state["impacts"]:
    print(f"{im.signal_id[:8]}  ->  {', '.join(im.affected_entities)}")

### After `forecast` — baseline vs risk-adjusted demand

In [ ]:
f = state["forecast"]
print("baseline:", f.baseline)
print("adjusted:", f.adjusted)
print(f.note)

### After `simulate` — quantified impact

In [ ]:
sim = state["simulation"]
print(f"stockout probability: {sim.stockout_probability:.0%}")
print(f"revenue impact:       {sim.revenue_impact:,.0f}")
print(f"assumptions:          {sim.assumptions}")

### After `recommend` — mitigation actions

In [ ]:
rec = state["recommendation"]
for action in rec.actions:
    print("-", action)
print(f"\n({rec.summary})")

## Where to go next

- Work on a single agent: `10_classify`, `15_news`, `20_impact`, `30_forecast`, `40_simulate`, `50_recommend`.
- Explore ingestion: `60_ingestion`.
- Onboarding & adding your own agent: `90_contributor_guide`.